# 第9章：适应（Adaptation）

## 概念

**适应** = 让 LLM 能够根据反馈进行自我改进

```
用户反馈 → [LLM] → 分析反馈 → 调整策略 → 改进输出
```

## 适应类型

| 类型 | 说明 | 示例 |
|------|------|------|
| **根据反馈调整** | 根据用户反馈改进回答 | 用户说"太长了"→缩短回答 |
| **自我修正** | 发现错误后自动修正 | 检查到语法错误→自动修正 |
| **环境适应** | 根据环境变化调整策略 | 时间变化→调整问候语 |

## 代码演示

使用 LangChain 实现三种适应方式

In [1]:
import os
from dotenv import load_dotenv
from langchain_openai import ChatOpenAI
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser

In [3]:
# 加载环境变量
load_dotenv()

# 使用小米 MiMo 模型
llm = ChatOpenAI(
    model="mimo-v2.5-pro",
    base_url="https://token-plan-cn.xiaomimimo.com/v1",
    api_key=os.environ.get("MIMO_API_KEY"),
    temperature=0
)

print("MiMo 模型初始化完成！")

MiMo 模型初始化完成！


## 方式1：根据反馈调整

根据用户反馈改进回答

In [ ]:
# 初始回答提示词
initial_prompt = ChatPromptTemplate.from_messages([
    ("system", "你是一个助手。请简洁回答用户问题。"),
    ("user", "{question}")
])

# 根据反馈调整的提示词
adapt_prompt = ChatPromptTemplate.from_messages([
    ("system", """你是一个助手。请根据用户反馈调整你的回答。

原始回答：{original_answer}







用户反馈：{feedback}

请根据反馈改进回答。"""),
    ("user", "请改进回答。")
])

# 创建链
initial_chain = initial_prompt | llm | StrOutputParser()
adapt_chain = adapt_prompt | llm | StrOutputParser()

# 定义适应函数
def answer_with_adaptation(question, feedback=None):
    # 生成初始回答
    answer = initial_chain.invoke({"question": question})
    
    # 如果有反馈，进行调整
    if feedback:
        answer = adapt_chain.invoke({
            "original_answer": answer,
            "feedback": feedback
        })
    
    return answer

# 测试对话
print("## 方式1：根据反馈调整 ##")

question = "什么是机器学习？"
print(f"\n用户问题: {question}")

answer = answer_with_adaptation(question)
print(f"\n初始回答: {answer}")

feedback = "太长了，请用一句话概括"
print(f"\n用户反馈: {feedback}")

adapted_answer = answer_with_adaptation(question, feedback)
print(f"\n调整后回答: {adapted_answer}")

## 方式1：根据反馈调整 ##

用户问题: 什么是机器学习？

初始回答: 机器学习是人工智能的一个分支，它让计算机能够通过**数据**和**经验**自动改进性能，而无需进行明确的编程。简单来说，就是让机器从数据中学习规律，并利用这些规律进行预测或决策。

**核心思想**：  
通过算法分析数据，识别模式，并基于这些模式做出判断或预测。

**举个例子**：  
- 你给机器学习模型提供大量猫和狗的图片，并告诉它哪些是猫、哪些是狗。  
- 模型会学习区分猫狗的特征（如耳朵形状、体型等）。  
- 之后，当你给它一张新图片时，它能自动判断是猫还是狗。

**主要类型**：  
1. **监督学习**：用带标签的数据训练（如分类、回归）。  
2. **无监督学习**：从无标签数据中发现隐藏模式（如聚类）。  
3. **强化学习**：通过试错和奖励机制学习策略（如游戏AI）。

**应用场景**：  
推荐系统（如抖音、淘宝）、语音助手、自动驾驶、医疗诊断等。

机器学习的目标是让机器具备“从数据中学习”的能力，从而解决复杂问题。

用户反馈: 太长了，请用一句话概括

调整后回答: 机器学习是让计算机通过数据自动发现规律，并利用这些规律进行预测或决策的技术。


## 方式2：自我修正

发现错误后自动修正

In [5]:
# 生成回答的提示词
generate_prompt = ChatPromptTemplate.from_messages([
    ("system", "你是一个助手。请回答用户问题。"),
    ("user", "{question}")
])

# 检查错误的提示词
check_prompt = ChatPromptTemplate.from_messages([
    ("system", """你是一个质量检查员。请检查回答是否有问题：
1. 语法错误
2. 逻辑错误
3. 事实错误

如果有问题，输出"有问题"和具体问题。
如果没有问题，输出"没有问题"。"""),
    ("user", "问题：{question}\n回答：{answer}")
])

# 修正回答的提示词
fix_prompt = ChatPromptTemplate.from_messages([
    ("system", """你是一个助手。请根据检查结果修正回答。

原始回答：{original_answer}
检查结果：{check_result}

请修正回答中的问题。"""),
    ("user", "请修正回答。")
])

# 创建链
generate_chain = generate_prompt | llm | StrOutputParser()
check_chain = check_prompt | llm | StrOutputParser()
fix_chain = fix_prompt | llm | StrOutputParser()

# 定义自我修正函数
def answer_with_self_correction(question, max_attempts=2):
    # 生成初始回答
    answer = generate_chain.invoke({"question": question})
    
    for attempt in range(max_attempts):
        # 检查回答
        check_result = check_chain.invoke({
            "question": question,
            "answer": answer
        })
        
        print(f"\n检查结果 (第{attempt + 1}次): {check_result}")
        
        # 如果没有问题，返回回答
        if "没有问题" in check_result:
            return answer
        
        # 修正回答
        answer = fix_chain.invoke({
            "original_answer": answer,
            "check_result": check_result
        })
        
        print(f"修正后回答: {answer}")
    
    return answer

# 测试对话
print("## 方式2：自我修正 ##")

question = "地球是平的吗？"
print(f"\n用户问题: {question}")

answer = answer_with_self_correction(question)
print(f"\n最终回答: {answer}")

## 方式2：自我修正 ##

用户问题: 地球是平的吗？

检查结果 (第1次): 没有问题

最终回答: ## 地球不是平的

地球是一个**近似球形**的天体（更准确地说，是一个两极稍扁、赤道略鼓的**椭球体**）。这是经过大量科学证据证实的事实：

### 主要证据包括：

1. **卫星照片**：从太空拍摄的照片清楚地显示地球是一个球体。
2. **环球航行**：麦哲伦船队早在16世纪就完成了环球航行，证明地球是圆的。
3. **地平线现象**：远处的船只总是先消失船身、最后消失桅杆，说明海面是弯曲的。
4. **月食**：月食时地球投在月球上的影子始终是圆形的。
5. **时区差异**：不同经度的地方在同一时刻有不同的白天和黑夜。
6. **重力**：大质量天体在自身引力作用下自然趋向球形。
7. **GPS和航空**：现代导航系统和航线都是基于球形地球模型运行的。

### 关于"地平说"

"地平说"（Flat Earth）是一种已被科学界**完全否定**的错误观点。虽然历史上某些古代文明曾认为大地是平的，但自古希腊时代起，科学家就已经认识到地球是球形的。

> **结论：地球是圆的（球形），不是平的。** 🌍


## 方式3：环境适应

根据环境变化调整策略

In [6]:
from datetime import datetime

# 获取当前时间
def get_time_context():
    hour = datetime.now().hour
    if hour < 12:
        return "早上"
    elif hour < 18:
        return "下午"
    else:
        return "晚上"

# 根据时间调整的提示词
time_prompt = ChatPromptTemplate.from_messages([
    ("system", """你是一个助手。请根据当前时间调整你的回答风格。

当前时间：{time_context}

如果是早上，使用活力充沛的语气。
如果是下午，使用专业稳重的语气。
如果是晚上，使用轻松友好的语气。"""),
    ("user", "{question}")
])

# 创建链
time_chain = time_prompt | llm | StrOutputParser()

# 定义环境适应函数
def answer_with_environment(question):
    # 获取环境信息
    time_context = get_time_context()
    
    # 调用 LLM
    response = time_chain.invoke({
        "time_context": time_context,
        "question": question
    })
    
    return response, time_context

# 测试对话
print("## 方式3：环境适应 ##")

question = "你好，今天过得怎么样？"
print(f"\n用户问题: {question}")

answer, time_context = answer_with_environment(question)
print(f"\n当前时间: {time_context}")
print(f"回答: {answer}")

## 方式3：环境适应 ##

用户问题: 你好，今天过得怎么样？

当前时间: 晚上
回答: 晚上好呀！✨ 今天过得挺充实的，一直在整理知识、帮人解答问题，像在图书馆里安静地翻书一样～ 你呢？今天有什么特别的事想分享吗？


## 三种适应方式对比

| 方式 | 触发条件 | 优点 | 缺点 | 适用场景 |
|------|----------|------|------|----------|
| 根据反馈调整 | 用户反馈 | 针对性强 | 需要用户反馈 | 交互式应用 |
| 自我修正 | 检查到错误 | 自动改进 | 可能过度修正 | 质量要求高 |
| 环境适应 | 环境变化 | 智能、自然 | 需要环境信息 | 上下文相关 |

## 适应的工作原理

```
输入/反馈/环境变化
    ↓
分析输入（理解反馈或环境）
    ↓
调整策略（决定如何改进）
    ↓
生成改进的回答
    ↓
输出改进后的结果
```

## 与其他模式的关系

| 第1章 提示链 | 第2章 路由 | 第3章 并行化 | 第4章 反思 | 第5章 工具使用 | 第6章 规划 | 第7章 多代理 | 第8章 记忆 | 第9章 适应 |
|-------------|-----------|-------------|-----------|--------------|----------|-------------|----------|----------|
| A → B → C | 选一条路 | A、B、C 同时执行 | 生成→批评→改进 | LLM + 外部工具 | 计划→执行 | 多个代理协作 | 保持对话状态 | 根据反馈改进 |
| 顺序执行 | 选择执行 | 并发执行 | 循环改进 | 扩展能力 | 结构化执行 | 分工合作 | 上下文保持 | 自我优化 |

## 实际应用场景

- **聊天机器人**：根据用户反馈调整回答风格
- **代码生成**：检查并修正生成的代码
- **内容创作**：根据时间、地点调整内容
- **客服系统**：根据用户情绪调整回答方式